# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [1]:
#@title 1.1 — Install
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu groq openai tqdm networkx spacy datasets langchain-community llama-index python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.0/165.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 

In [20]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

# Load .env nếu chạy local
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option('display.max_colwidth', 120)

def get_secret(name, default=None):
    """
    Lấy secret ưu tiên: (1) Google Colab Secrets (userdata) -> (2) Biến môi trường / .env -> (3) default value
    Không gây lỗi SecretNotFoundError hay ModuleNotFoundError.
    """
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None and str(value).strip() != '':
            return str(value).strip()
    except Exception:
        pass
    val = os.environ.get(name)
    if val is not None and str(val).strip() != '':
        return str(val).strip()
    return default

# Cấu hình Secrets
NEO4J_URI = get_secret('NEO4J_URI', '')
NEO4J_USER = get_secret('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = get_secret('NEO4J_PASSWORD', '')
NEO4J_DATABASE = get_secret('NEO4J_DATABASE', 'neo4j')

GROQ_API_KEY = get_secret('GROQ_API_KEY', '')
GROQ_MODEL = get_secret('GROQ_MODEL', 'llama-3.3-70b-versatile')

JUDGE_PROVIDER = get_secret('JUDGE_PROVIDER', 'groq').lower()
JUDGE_MODEL = get_secret('JUDGE_MODEL', 'llama-3.3-70b-versatile')
OPENAI_API_KEY = get_secret('OPENAI_API_KEY', '')
HF_TOKEN = get_secret('HF_TOKEN', '')
OPENAI_API_KEY = get_secret('OPENAI_API_KEY', '')

# Đường dẫn dữ liệu tương thích cả Google Colab (/content) và Local
DATA_PATH = '/content/hackernoon_subset.csv' if Path('/content/hackernoon_subset.csv').exists() else 'data/hackernoon_subset.csv'
LAB_MAX_ARTICLES = 5000   # Khớp với tập 5000 bài báo của graphrag_golden_50_first5000
LAB_MAX_CHUNKS = 8000
EXTRACTION_MAX_CHUNKS = 600
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [3]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "/content/hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 1_000_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

Đang kết nối luồng dữ liệu (streaming)...


README.md:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Đang ghi dữ liệu vào: /content/hackernoon_subset.csv


Đang tải (MB):   0%|          | 0/300 [00:00<?, ?MB/s]


[DỪNG] Đã đạt giới hạn dung lượng: 300.00 MB (Tổng: 514,417 dòng)
✅ Hoàn thành: /content/hackernoon_subset.csv
   Rows: 514,417
   Size: 300.00 MB


In [4]:
#@title 1.4 — Neo4j connection + schema
# Lấy thông tin kết nối an toàn qua get_secret (tương thích Colab userdata và .env)
NEO4J_URI = get_secret('NEO4J_URI', NEO4J_URI)
NEO4J_USER = get_secret('NEO4J_USER', NEO4J_USER or 'neo4j')
NEO4J_PASSWORD = get_secret('NEO4J_PASSWORD', NEO4J_PASSWORD)
NEO4J_DATABASE = get_secret('NEO4J_DATABASE', NEO4J_DATABASE or 'neo4j')
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError(
            "Thiếu Neo4j secrets! Vui lòng thiết lập 'NEO4J_URI' và 'NEO4J_PASSWORD' "
            "trong Colab Secrets (biểu tượng chìa khóa 🔑 ở thanh menu trái) hoặc file .env."
        )
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected successfully.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()


✅ Neo4j connected successfully.
✅ Schema ready.


In [5]:
#@title 1.5 — Loader + exact dedup + chunking

def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()


def sha1(x):
    return hashlib.sha1(
        str(x).encode("utf-8", errors="ignore")
    ).hexdigest()


def pick_col(df, candidates, required=True):
    # Chuẩn hóa tên cột để tìm không phân biệt hoa/thường + khoảng trắng
    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for c in candidates:
        key = c.strip().lower()
        if key in lookup:
            return lookup[key]

    if required:
        raise KeyError(
            f"Không tìm thấy cột phù hợp.\n"
            f"Cần một trong: {candidates}\n"
            f"Các cột hiện có: {list(df.columns)}"
        )

    return None


def load_news(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)

    if suffix == ".json":
        return pd.read_json(path)

    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)

    raise ValueError(f"Unsupported file type: {suffix}")


def standardize_news(raw):

    print("📌 Dataset columns:")
    print(list(raw.columns))

    # -------------------------
    # 1. Tìm cột text
    # -------------------------
    text_col = pick_col(
        raw,
        [
            "text",
            "content",
            "article",
            "body",
            "story",

            # thêm các tên thường gặp
            "description",
            "article_text",
            "full_text",
            "news_text",
            "summary",
            "abstract",
        ],
        required=True
    )

    # -------------------------
    # 2. Tìm cột title
    # -------------------------
    title_col = pick_col(
        raw,
        [
            "title",
            "headline",
            "heading",
            "news_title",
        ],
        required=False
    )

    # -------------------------
    # 3. Tìm cột ngày
    # -------------------------
    date_col = pick_col(
        raw,
        [
            "published_date",
            "date",
            "published_at",
            "created_at",
            "publication_date",
            "publish_date",
            "timestamp",
        ],
        required=False
    )

    # -------------------------
    # 4. Tìm cột ID
    # -------------------------
    id_col = pick_col(
        raw,
        [
            "id",
            "article_id",
            "story_id",
            "uuid",
            "news_id",
            "doc_id",
        ],
        required=False
    )

    print("\n✅ Column mapping:")
    print("text_col :", text_col)
    print("title_col:", title_col)
    print("date_col :", date_col)
    print("id_col   :", id_col)

    # -------------------------
    # 5. Chuẩn hóa DataFrame
    # -------------------------
    df = pd.DataFrame()

    df["text"] = (
        raw[text_col]
        .fillna("")
        .map(norm_space)
    )

    if title_col:
        df["title"] = (
            raw[title_col]
            .fillna("")
            .map(norm_space)
        )
    else:
        df["title"] = ""

    # -------------------------
    # 6. Chuẩn hóa ngày
    # -------------------------
    if date_col:
        df["published_date"] = (
            pd.to_datetime(
                raw[date_col],
                errors="coerce",
                utc=True
            )
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    # -------------------------
    # 7. Article ID
    # -------------------------
    if id_col:
        df["article_id"] = (
            raw[id_col]
            .fillna("")
            .astype(str)
        )
    else:
        df["article_id"] = [
            sha1(f"{title}\n{text}")[:20]
            for title, text in zip(
                df["title"],
                df["text"]
            )
        ]

    # -------------------------
    # 8. Bỏ bài quá ngắn
    # -------------------------
    before_short_filter = len(df)

    df = (
        df[df["text"].str.len() >= 80]
        .copy()
        .reset_index(drop=True)
    )

    print(
        f"Length filter: "
        f"{before_short_filter:,} -> {len(df):,}"
    )

    # -------------------------
    # 9. Exact dedup
    # -------------------------
    df["dedup_key"] = [
        sha1(
            norm_space(
                f"{title}\n{text}"
            ).lower()
        )
        for title, text in zip(
            df["title"],
            df["text"]
        )
    ]

    before_dedup = len(df)

    df = (
        df
        .drop_duplicates("dedup_key")
        .drop(columns="dedup_key")
        .reset_index(drop=True)
    )

    print(
        f"Exact dedup: "
        f"{before_dedup:,} -> {len(df):,}"
    )

    # -------------------------
    # 10. Giới hạn số article
    # -------------------------
    if (
        LAB_MAX_ARTICLES is not None
        and LAB_MAX_ARTICLES > 0
        and len(df) > LAB_MAX_ARTICLES
    ):
        df = (
            df
            .iloc[:LAB_MAX_ARTICLES]
            .reset_index(drop=True)
        )

        print(
            f"Article limit: "
            f"{len(df):,}/{LAB_MAX_ARTICLES:,}"
        )

    return df


def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()

    if not words:
        return []

    step = max(1, size - overlap)

    out = []

    for start in range(0, len(words), step):

        part = words[start:start + size]

        if not part:
            break

        out.append(" ".join(part))

        if start + size >= len(words):
            break

    return out


def build_chunks(news_df):

    rows = []

    for r in tqdm(
        news_df.itertuples(index=False),
        total=len(news_df),
        desc="Chunking"
    ):

        chunks = chunk_text(
            r.text,
            CHUNK_WORDS,
            CHUNK_OVERLAP_WORDS
        )

        for i, text in enumerate(chunks):

            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })

            if (
                LAB_MAX_CHUNKS is not None
                and LAB_MAX_CHUNKS > 0
                and len(rows) >= LAB_MAX_CHUNKS
            ):
                return pd.DataFrame(rows)

    return pd.DataFrame(rows)


# =========================================================
# RUN
# =========================================================

# Tìm dataset từ các đường dẫn trên Colab và Local
data_candidates = [
    DATA_PATH,
    '/content/hackernoon_subset.csv',
    'data/hackernoon_subset.csv',
    'hackernoon_subset.csv'
]
actual_data_path = None
for p in data_candidates:
    if p and Path(p).exists():
        actual_data_path = p
        break
if actual_data_path is None:
    raise FileNotFoundError(f"Không tìm thấy dataset. Hãy chạy Cell 1.3 trước hoặc tải dataset vào {data_candidates}")
raw_df = load_news(actual_data_path)

print("Raw shape:", raw_df.shape)

news_df = standardize_news(raw_df)

print("Standardized news:", news_df.shape)

chunks_df = build_chunks(news_df)

print("Chunks:", chunks_df.shape)

display(news_df.head(3))
display(chunks_df.head(3))

Raw shape: (514417, 7)
📌 Dataset columns:
['companyName', 'companyUrl', 'published_at', 'url', 'title', 'main_image', 'description']

✅ Column mapping:
text_col : description
title_col: title
date_col : published_at
id_col   : None
Length filter: 514,417 -> 245,324
Exact dedup: 245,324 -> 212,212
Article limit: 5,000/5,000
Standardized news: (5000, 4)


Chunking:   0%|          | 0/5000 [00:00<?, ?it/s]

Chunks: (5004, 5)


,text,title,published_date,article_id
0,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,1a05beb7aa3071be6fd7
1,ELKO — An eighth grader at Adobe Middle School is one of 34 middle school aged girls in Nevada to be recognized by t...,Adobe student receives national Information and Technology award,2023-05-02,ec98609611765e97440d
2,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,8e922bc62b578e73e815


,chunk_id,article_id,title,published_date,text
0,1a05beb7aa3071be6fd7::c0000,1a05beb7aa3071be6fd7,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...
1,ec98609611765e97440d::c0000,ec98609611765e97440d,Adobe student receives national Information and Technology award,2023-05-02,ELKO — An eighth grader at Adobe Middle School is one of 34 middle school aged girls in Nevada to be recognized by t...
2,8e922bc62b578e73e815::c0000,8e922bc62b578e73e815,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [6]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
GROQ_API_KEY = get_secret('GROQ_API_KEY', GROQ_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    global groq_client
    if groq_client is None:
        api_key = get_secret('GROQ_API_KEY', GROQ_API_KEY)
        if api_key:
            groq_client = Groq(api_key=api_key)
        else:
            raise RuntimeError("Thiếu GROQ_API_KEY. Vui lòng thêm 'GROQ_API_KEY' vào Colab Secrets 🔑 hoặc file .env.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [7]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

Coref:   0%|          | 0/120 [00:00<?, ?it/s]

# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [8]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

    return pd.DataFrame(triples), pd.DataFrame(errors)

raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
display(raw_triples_df.head())

NER+RE:   0%|          | 0/150 [00:00<?, ?it/s]

,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Prospect Medical Holdings,Company,DEVELOPED,technology,Technology,916a3ebf144f682693af::c0000,2023-08-05,code and develop the technology for these systems,0.0
1,Patrick Racz,Person,DEVELOPED,Smartflash,Technology,198660b5d2a419ac68f2::c0000,2023-04-17,Inventor Patrick Racz claims his Smartflash technology is at the heart of the ...,1.0
2,Symbotic,Company,USES,automation technology,Technology,741f701c5971b540c917::c0000,2023-08-17,Symbotic aims to revolutionize warehouse operations through automation technology.,1.0
3,Infosys,Company,USES,innovative technologies,Technology,486b82396c59013f41d9::c0000,2023-05-17,Infosys positioned as the leading service provider that can drive transformational change by using innovative techno...,1.0


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [9]:
#@title 2.2 — Entity resolution (robust column mapping)

from collections import Counter, defaultdict
from difflib import SequenceMatcher
import unicodedata
import re
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer


# ============================================================
# 0. CHUẨN HÓA COLUMN CỦA raw_triples_df
# ============================================================

def find_column(df, candidates, required=True):
    """
    Tìm column theo nhiều tên có thể xuất hiện.
    Không phân biệt hoa/thường.
    """
    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.lower()

        if key in lookup:
            return lookup[key]

    if required:
        raise KeyError(
            f"Không tìm thấy một trong các cột {candidates}\n"
            f"Các cột hiện tại: {df.columns.tolist()}"
        )

    return None


def normalize_triple_columns(raw_df):
    """
    Đưa DataFrame về schema chuẩn:

    source_raw
    source_type
    target_raw
    target_type
    """

    df = raw_df.copy()

    print("📌 raw_triples_df columns:")
    print(df.columns.tolist())

    source_col = find_column(
        df,
        [
            "source_raw",
            "source",
            "subject",
            "head",
            "entity1",
            "source_name",
        ],
    )

    source_type_col = find_column(
        df,
        [
            "source_type",
            "subject_type",
            "head_type",
            "entity1_type",
            "source_label",
        ],
    )

    target_col = find_column(
        df,
        [
            "target_raw",
            "target",
            "object",
            "tail",
            "entity2",
            "target_name",
        ],
    )

    target_type_col = find_column(
        df,
        [
            "target_type",
            "object_type",
            "tail_type",
            "entity2_type",
            "target_label",
        ],
    )

    rename_map = {
        source_col: "source_raw",
        source_type_col: "source_type",
        target_col: "target_raw",
        target_type_col: "target_type",
    }

    df = df.rename(columns=rename_map)

    # Ép về string để tránh lỗi NaN / float
    for col in [
        "source_raw",
        "source_type",
        "target_raw",
        "target_type",
    ]:
        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
            .map(norm_space)
        )

    # Bỏ triple thiếu source/target
    before = len(df)

    df = df[
        (df["source_raw"] != "")
        & (df["target_raw"] != "")
        & (df["source_type"] != "")
        & (df["target_type"] != "")
    ].copy()

    df = df.reset_index(drop=True)

    print("\n✅ Triple column mapping:")
    print("source_raw :", source_col)
    print("source_type:", source_type_col)
    print("target_raw :", target_col)
    print("target_type:", target_type_col)

    print(
        f"\nValid triples: {before:,} -> {len(df):,}"
    )

    return df


# Chuẩn hóa ngay từ đầu
raw_triples_df = normalize_triple_columns(raw_triples_df)


# ============================================================
# 1. ENTITY NORMALIZATION
# ============================================================

CORP_SUFFIXES = {
    "inc",
    "incorporated",
    "corp",
    "corporation",
    "ltd",
    "limited",
    "llc",
    "plc",
    "co",
    "company",
}


MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",

    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",

    "meta platforms": "Meta",
    "meta platforms inc": "Meta",

    "aapl": "Apple",
    "apple inc": "Apple",
}


def norm_entity(name):
    s = unicodedata.normalize(
        "NFKC",
        norm_space(name)
    ).lower()

    s = re.sub(
        r"[^\w\s\-\.]",
        " ",
        s
    )

    return re.sub(
        r"\s+",
        " ",
        s
    ).strip()


def strip_suffix(name):
    toks = (
        norm_entity(name)
        .replace(".", "")
        .split()
    )

    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()

    return " ".join(toks)


def merge_guard(a, b):
    """
    Chỉ merge khi tên đủ giống nhau.
    """

    na = strip_suffix(a)
    nb = strip_suffix(b)

    if not na or not nb:
        return False

    if na == nb:
        return True

    return (
        SequenceMatcher(
            None,
            na,
            nb
        ).ratio()
        >= 0.72
    )


# ============================================================
# 2. EMBEDDING MODEL
# ============================================================

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = None


def get_embedder():
    global embedder

    if embedder is None:
        print(
            f"⏳ Loading embedding model: "
            f"{EMBED_MODEL}"
        )

        embedder = SentenceTransformer(
            EMBED_MODEL
        )

        print("✅ Embedding model loaded.")

    return embedder


# ============================================================
# 3. UNION FIND
# ============================================================

class UF:

    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):

        if self.p[x] != x:
            self.p[x] = self.find(
                self.p[x]
            )

        return self.p[x]

    def union(self, a, b):

        a = self.find(a)
        b = self.find(b)

        if a != b:
            self.p[b] = a


# ============================================================
# 4. BUILD ENTITY RESOLUTION MAP
# ============================================================

def build_resolution_map(
    raw_triples_df,
    threshold=0.90,
    top_k=5,
):

    required = [
        "source_raw",
        "source_type",
        "target_raw",
        "target_type",
    ]

    missing = [
        c for c in required
        if c not in raw_triples_df.columns
    ]

    if missing:
        raise KeyError(
            f"Thiếu columns: {missing}\n"
            f"Hiện có: {raw_triples_df.columns.tolist()}"
        )

    # --------------------------------
    # Collect mentions
    # --------------------------------

    mentions = []

    for r in raw_triples_df.itertuples(
        index=False
    ):

        mentions.append(
            (
                r.source_type,
                r.source_raw
            )
        )

        mentions.append(
            (
                r.target_type,
                r.target_raw
            )
        )

    if not mentions:
        return {}, pd.DataFrame(
            columns=[
                "type",
                "left",
                "right",
                "similarity",
                "decision",
            ]
        )

    # --------------------------------
    # Count entity mentions
    # --------------------------------

    counts = Counter(
        (
            entity_type,
            norm_entity(name)
        )
        for entity_type, name in mentions
    )

    display_name = {}

    for entity_type, name in mentions:

        key = (
            entity_type,
            norm_entity(name)
        )

        display_name.setdefault(
            key,
            name
        )

    mapping = {}
    audit = []

    # --------------------------------
    # Manual aliases
    # --------------------------------

    for key in counts:

        entity_type, normalized = key

        if normalized in MANUAL_ALIASES:

            canonical = MANUAL_ALIASES[
                normalized
            ]

            mapping[key] = canonical

            audit.append({
                "type": entity_type,
                "left": display_name[key],
                "right": canonical,
                "similarity": 1.0,
                "decision": "MERGE_MANUAL",
            })

    # --------------------------------
    # Không phụ thuộc ALLOWED_NODE_TYPES
    # Lấy type trực tiếp từ dataset
    # --------------------------------

    node_types = sorted(
        set(
            raw_triples_df[
                "source_type"
            ].tolist()
        )
        |
        set(
            raw_triples_df[
                "target_type"
            ].tolist()
        )
    )

    print(
        "📌 Entity types:",
        node_types
    )

    # --------------------------------
    # Vector resolution theo từng type
    # --------------------------------

    for typ in node_types:

        keys = [
            k
            for k in counts
            if k[0] == typ
            and k not in mapping
        ]

        if not keys:
            continue

        names = [
            display_name[k]
            for k in keys
        ]

        print(
            f"🔎 Resolving {typ}: "
            f"{len(names):,} entities"
        )

        # Chỉ có 1 entity thì không cần FAISS
        if len(names) == 1:

            mapping[keys[0]] = names[0]

            continue

        vecs = (
            get_embedder()
            .encode(
                names,
                batch_size=128,
                show_progress_bar=False,
                normalize_embeddings=True,
            )
            .astype("float32")
        )

        index = faiss.IndexFlatIP(
            vecs.shape[1]
        )

        index.add(vecs)

        k = min(
            max(2, top_k),
            len(names)
        )

        sims, nbrs = index.search(
            vecs,
            k
        )

        uf = UF(
            len(names)
        )

        # --------------------------------
        # Candidate pairs
        # --------------------------------

        for i in range(len(names)):

            for score, j in zip(
                sims[i],
                nbrs[i]
            ):

                j = int(j)
                score = float(score)

                if j < 0:
                    continue

                # tránh self-pair + duplicate pair
                if i >= j:
                    continue

                if score < threshold:
                    continue

                ok = merge_guard(
                    names[i],
                    names[j]
                )

                audit.append({
                    "type": typ,
                    "left": names[i],
                    "right": names[j],
                    "similarity": score,
                    "decision": (
                        "MERGE_VECTOR"
                        if ok
                        else "REJECT_GUARD"
                    ),
                })

                if ok:
                    uf.union(
                        i,
                        j
                    )

        # --------------------------------
        # Build groups
        # --------------------------------

        groups = defaultdict(list)

        for i in range(len(names)):

            root = uf.find(i)

            groups[root].append(i)

        # --------------------------------
        # Chọn canonical entity
        # --------------------------------

        for idxs in groups.values():

            best = sorted(
                idxs,
                key=lambda i: (
                    -counts[keys[i]],
                    len(names[i]),
                    names[i].lower(),
                ),
            )[0]

            canonical = names[best]

            for i in idxs:

                mapping[
                    keys[i]
                ] = canonical

    # --------------------------------
    # Entity chưa map -> giữ nguyên
    # --------------------------------

    for key in counts:

        mapping.setdefault(
            key,
            display_name[key]
        )

    audit_df = pd.DataFrame(
        audit
    )

    return mapping, audit_df


# ============================================================
# 5. CANONICALIZE TRIPLES
# ============================================================

def canonicalize_triples(
    raw_df,
    mapping,
):

    df = raw_df.copy()

    required = [
        "source_raw",
        "source_type",
        "target_raw",
        "target_type",
    ]

    missing = [
        c
        for c in required
        if c not in df.columns
    ]

    if missing:
        raise KeyError(
            f"canonicalize_triples thiếu: "
            f"{missing}\n"
            f"Available: {df.columns.tolist()}"
        )

    def canon(name, typ):

        n = norm_entity(name)

        return mapping.get(
            (typ, n),
            MANUAL_ALIASES.get(
                n,
                name
            )
        )

    # Quan trọng:
    # dùng df["column"] thay vì df.column
    df["source_name"] = [
        canon(name, typ)
        for name, typ in zip(
            df["source_raw"],
            df["source_type"],
        )
    ]

    df["target_name"] = [
        canon(name, typ)
        for name, typ in zip(
            df["target_raw"],
            df["target_type"],
        )
    ]

    # --------------------------------
    # Normalized name
    # --------------------------------

    df["source_name_norm"] = (
        df["source_name"]
        .map(norm_entity)
    )

    df["target_name_norm"] = (
        df["target_name"]
        .map(norm_entity)
    )

    # --------------------------------
    # Stable IDs
    # --------------------------------

    df["source_id"] = [
        sha1(
            f"{typ}:{name}"
        )[:24]
        for typ, name in zip(
            df["source_type"],
            df["source_name_norm"],
        )
    ]

    df["target_id"] = [
        sha1(
            f"{typ}:{name}"
        )[:24]
        for typ, name in zip(
            df["target_type"],
            df["target_name_norm"],
        )
    ]

    # Bỏ self-loop
    df = df[
        df["source_id"]
        != df["target_id"]
    ].copy()

    return df.reset_index(
        drop=True
    )


# ============================================================
# 6. RUN ENTITY RESOLUTION
# ============================================================

print("\n" + "=" * 60)
print("ENTITY RESOLUTION")
print("=" * 60)

entity_map, entity_resolution_audit_df = (
    build_resolution_map(
        raw_triples_df
    )
)

triples_df = canonicalize_triples(
    raw_triples_df,
    entity_map
)

print("\n✅ Entity resolution completed")

print(
    "Raw triples:",
    len(raw_triples_df)
)

print(
    "Canonical triples:",
    len(triples_df)
)

print(
    "Entity mappings:",
    len(entity_map)
)

print(
    "Resolution decisions:",
    len(entity_resolution_audit_df)
)


print("\n📌 Resolution audit:")
display(
    entity_resolution_audit_df.head(20)
)


print("\n📌 Canonical triples:")
display(
    triples_df.head(20)
)

📌 raw_triples_df columns:
['source_raw', 'source_type', 'relation', 'target_raw', 'target_type', 'source_chunk_id', 'published_date', 'evidence', 'confidence']

✅ Triple column mapping:
source_raw : source_raw
source_type: source_type
target_raw : target_raw
target_type: target_type

Valid triples: 4 -> 4

ENTITY RESOLUTION
📌 Entity types: ['Company', 'Person', 'Technology']
🔎 Resolving Company: 3 entities
⏳ Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded.
🔎 Resolving Person: 1 entities
🔎 Resolving Technology: 4 entities

✅ Entity resolution completed
Raw triples: 4
Canonical triples: 4
Entity mappings: 8
Resolution decisions: 0

📌 Resolution audit:


""



📌 Canonical triples:


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence,source_name,target_name,source_name_norm,target_name_norm,source_id,target_id
0,Prospect Medical Holdings,Company,DEVELOPED,technology,Technology,916a3ebf144f682693af::c0000,2023-08-05,code and develop the technology for these systems,0.0,Prospect Medical Holdings,technology,prospect medical holdings,technology,d50f81cd7c5de4c7596bf31e,3669130c05135dd501f33457
1,Patrick Racz,Person,DEVELOPED,Smartflash,Technology,198660b5d2a419ac68f2::c0000,2023-04-17,Inventor Patrick Racz claims his Smartflash technology is at the heart of the ...,1.0,Patrick Racz,Smartflash,patrick racz,smartflash,c3bd261e79278b4d6b2b95db,51532d47cc7852732127fe50
2,Symbotic,Company,USES,automation technology,Technology,741f701c5971b540c917::c0000,2023-08-17,Symbotic aims to revolutionize warehouse operations through automation technology.,1.0,Symbotic,automation technology,symbotic,automation technology,f5e016a5966ce41aa7ef0f02,a3da966c5bd4490880d33191
3,Infosys,Company,USES,innovative technologies,Technology,486b82396c59013f41d9::c0000,2023-05-17,Infosys positioned as the leading service provider that can drive transformational change by using innovative techno...,1.0,Infosys,innovative technologies,infosys,innovative technologies,26fe01fbff18a0ced2a65600,8711171b850510249770b7b7


In [10]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

# nodes_df = build_nodes(triples_df)
# bulk_insert_nodes(nodes_df)
# bulk_insert_edges(triples_df)

In [11]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

# graph_counts, top_degree_df = graph_checks()

# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [12]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

# build_flat_index(chunks_df)

## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [13]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

# build_entity_matcher(nodes_df)

In [14]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [15]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GROQ_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [16]:
#@title 4.1 — Golden Dataset (Nạp tự động graphrag_golden_50_first5000.csv)
GOLDEN_CANDIDATE_PATHS = [
    'data/graphrag_golden_50_first5000.csv',
    'data/graphrag_golden_50_first5000_detailed.csv',
    '/content/graphrag_golden_50_first5000.csv',
    '/content/data/graphrag_golden_50_first5000.csv',
    'graphrag_golden_50_first5000.csv',
    'data/golden_dataset.csv',
    '/content/golden_dataset.csv',
]

golden_df = None
loaded_file = None
for p_str in GOLDEN_CANDIDATE_PATHS:
    p = Path(p_str)
    if p.exists():
        golden_df = pd.read_csv(p)
        loaded_file = str(p)
        print(f'✅ Đã tải thành công Golden Dataset từ: {loaded_file}')
        print(f'📊 Tổng số câu hỏi: {len(golden_df)} câu')
        break

if golden_df is None:
    print('⚠️ Không tìm thấy file CSV, dùng dataset mẫu dự phòng:')
    golden_df = pd.DataFrame([
        {'id':'G01','group':'factoid','question':'Who was the CEO of Hugging Face in 2023?','reference_answer':'Clément Delangue','reference_evidence':'Validate against instructor dump.'},
    ])

# Tuỳ chọn giới hạn số câu để test nhanh nếu muốn (đặt None để chạy full 50 câu)
EVAL_SAMPLE_N = None  # ví dụ: 5 hoặc 10 để test nhanh, None để chạy toàn bộ 50 câu
if EVAL_SAMPLE_N and len(golden_df) > EVAL_SAMPLE_N:
    print(f'⚡ Chế độ Test nhanh: Lấy {EVAL_SAMPLE_N}/{len(golden_df)} câu hỏi')
    eval_golden_df = golden_df.head(EVAL_SAMPLE_N).copy()
else:
    eval_golden_df = golden_df.copy()

def validate_golden(df, require_answers=True):
    required = {'id','group','question','reference_answer'}
    if not required.issubset(df.columns):
        raise ValueError(f'Missing columns: {required-set(df.columns)}')
    if require_answers and df.reference_answer.fillna('').str.strip().eq('').any():
        display(df[df.reference_answer.fillna('').str.strip().eq('')][['id','question']])
        raise ValueError('Điền reference_answer trước final evaluation.')
    print(f'✅ Golden Dataset valid ({len(df)} câu hỏi hợp lệ).')

validate_golden(eval_golden_df, require_answers=True)
display(eval_golden_df.head())


⚠️ Không tìm thấy file CSV, dùng dataset mẫu dự phòng:
✅ Golden Dataset valid (1 câu hỏi hợp lệ).


,id,group,question,reference_answer,reference_evidence
0,G01,factoid,Who was the CEO of Hugging Face in 2023?,Clément Delangue,Validate against instructor dump.


In [17]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [22]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = 'outputs/graphrag_eval_checkpoint.csv'
os.makedirs('outputs', exist_ok=True)
os.makedirs('/content/outputs', exist_ok=True)

def run_evaluation(golden_df):
    rows = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc='Evaluation'):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat['answer'], flat['context'])
        jg = judge_answer(q.question, q.reference_answer, graph['answer'], graph['context'])

        rows.append({
            'id':q.id, 'group':q.group, 'question':q.question,
            'reference_answer':q.reference_answer,
            'flat_answer':flat['answer'], 'graph_answer':graph['answer'],
            'flat_comprehensiveness':jf['comprehensiveness'],
            'graph_comprehensiveness':jg['comprehensiveness'],
            'flat_faithfulness':jf['faithfulness'],
            'graph_faithfulness':jg['faithfulness'],
            'flat_multi_hop_reasoning':jf['multi_hop_reasoning'],
            'graph_multi_hop_reasoning':jg['multi_hop_reasoning'],
            'flat_latency_s':flat['latency_s'],
            'graph_latency_s':graph['latency_s'],
            'flat_total_tokens':flat.get('total_tokens'),
            'graph_total_tokens':graph.get('total_tokens'),
            'flat_judge_rationale':jf['rationale'],
            'graph_judge_rationale':jg['rationale'],
            'graph_supernode_events':len(
                graph['graph_debug']['diagnostics'].get('supernode_events',[])
            )
        })
        temp_df = pd.DataFrame(rows)
        for cp in [CHECKPOINT, '/content/graphrag_eval_checkpoint.csv', '/content/outputs/graphrag_eval_checkpoint.csv']:
            try:
                temp_df.to_csv(cp, index=False)
            except Exception:
                pass
    return pd.DataFrame(rows)

# Chạy benchmark đánh giá:
eval_results_df = run_evaluation(eval_golden_df)
display(eval_results_df.head())


Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G01,factoid,Who was the CEO of Hugging Face in 2023?,Clément Delangue,The CEO of Hugging Face in 2023 was **Evan Gappelberg**【e8bdde6f283b33301ffa=c0000】.,The CEO of Hugging Face in 2023 was **Evan Gappelberg**【e8bdde6f283b33301ffa::c0000】.,1,1,1,1,1,1,0.459698,0.202883,749,556,"The candidate answer incorrectly states that Evan Gappelberg is the CEO of Hugging Face in 2023, while the reference...","The candidate answer incorrectly states that Evan Gappelberg is the CEO of Hugging Face in 2023, while the reference...",0


In [21]:
import os

# 1. Initialize Flat RAG index (already built, but ensures global state)
print("Verifying Flat RAG index...")
if flat_index is None:
    build_flat_index(chunks_df)

# 2. Bulk insert data into Neo4j (already inserted, but ensures nodes_df exists)
print("Checking Neo4j data...")
nodes_df = build_nodes(triples_df)

# 3. Build entity matcher
print("Building entity matcher...")
build_entity_matcher(nodes_df)

# 4. Now run evaluation
print("Starting evaluation benchmark...")
eval_results_df = run_evaluation(eval_golden_df)
display(eval_results_df.head())

Verifying Flat RAG index...
Checking Neo4j data...
Building entity matcher...
Starting evaluation benchmark...


Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G01,factoid,Who was the CEO of Hugging Face in 2023?,Clément Delangue,The CEO of Hugging Face in 2023 was **Evan Gappelberg**【e8bdde6f283b33301ffa=c0000】.,The CEO of Hugging Face in 2023 was **Evan Gappelberg**【e8bdde6f283b33301ffa::c0000】.,1,1,1,1,1,1,0.361546,0.221335,749,556,"The candidate answer incorrectly states that Evan Gappelberg is the CEO of Hugging Face in 2023, while the reference...","The candidate answer incorrectly states that Evan Gappelberg is the CEO of Hugging Face in 2023, while the reference...",0


In [23]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        'Comprehensiveness':('flat_comprehensiveness','graph_comprehensiveness'),
        'Faithfulness':('flat_faithfulness','graph_faithfulness'),
        'Multi-hop reasoning':('flat_multi_hop_reasoning','graph_multi_hop_reasoning'),
        'Latency (s)':('flat_latency_s','graph_latency_s'),
        'Token usage':('flat_total_tokens','graph_total_tokens'),
    }

    rows = []
    for group, g in eval_df.groupby('group'):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors='coerce').mean()
            gr = pd.to_numeric(g[gc], errors='coerce').mean()

            if metric in {'Latency (s)','Token usage'}:
                comment = 'Flat RAG thường rẻ/nhanh hơn.' if f < gr else 'GraphRAG không đắt hơn trong sample này.'
            else:
                delta = gr - f
                if delta >= .75:
                    comment = 'GraphRAG cải thiện rõ; kiểm tra rationale và provenance.'
                elif delta <= -.5:
                    comment = 'Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu.'
                else:
                    comment = 'Hai phương pháp gần nhau.'

            rows.append({
                'Loại câu hỏi':group, 'Metric':metric,
                'Flat RAG':round(f,3) if pd.notna(f) else np.nan,
                'GraphRAG':round(gr,3) if pd.notna(gr) else np.nan,
                'Nhận xét phân tích':comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)

# Export ra các đường dẫn tương thích
for eval_out in ['outputs/graphrag_eval_results.csv', '/content/graphrag_eval_results.csv', '/content/outputs/graphrag_eval_results.csv']:
    try:
        eval_results_df.to_csv(eval_out, index=False)
    except Exception:
        pass

for sum_out in ['outputs/graphrag_vs_flatrag_summary.csv', '/content/graphrag_vs_flatrag_summary.csv', '/content/outputs/graphrag_vs_flatrag_summary.csv']:
    try:
        comparison_df.to_csv(sum_out, index=False)
    except Exception:
        pass
print('✅ Đã xuất kết quả thành công vào outputs/graphrag_eval_results.csv và outputs/graphrag_vs_flatrag_summary.csv')


,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,factoid,Comprehensiveness,1.00,1.000,Hai phương pháp gần nhau.
1,factoid,Faithfulness,1.00,1.000,Hai phương pháp gần nhau.
2,factoid,Multi-hop reasoning,1.00,1.000,Hai phương pháp gần nhau.
3,factoid,Latency (s),0.46,0.203,GraphRAG không đắt hơn trong sample này.
4,factoid,Token usage,749.00,556.000,GraphRAG không đắt hơn trong sample này.


✅ Đã xuất kết quả thành công vào outputs/graphrag_eval_results.csv và outputs/graphrag_vs_flatrag_summary.csv


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [24]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)

{'id': '1436a8eb4a4e8197fa650e48', 'name': 'Opsys Tech', 'degree': 3} fetched= 3
No audit rows.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [ ]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

# community_df = build_communities()

In [ ]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau